<center><img src="../../image/dice.png" alt="DICE" style="width: 300px;"/></center>

# 6.机械臂抓取区域标定

@－－－－DICE－－－－<br>
@  文件名：6.机械臂抓取区域标定.ipynb <br>
@  版本：V2.0 <br>
@  说明：Jetson Nano 机械臂基本运动<br>
本实验属于机械臂基础运动实验部分<br>

## 导入依赖

In [2]:
import math
import numpy as np
import yaml
# 自定义库 
from transform import Transform
from icp import icp_solver_svd
from arm5dof_uservo import Arm5DoFUServo
from config import *

## 载入九点标定的配置文件

In [3]:
# 载入九点标定法的配置文件
handeye_9points_config = None
with open("config/handeye_9points.yaml", 'r', encoding='utf-8') as f:
    handeye_9points_config = yaml.load(f.read(), Loader=yaml.SafeLoader)

## 创建机械臂

In [4]:
# 创建机械臂
arm = Arm5DoFUServo(is_init_pose=False)

In [5]:
# 设置为阻尼模式
arm.set_damping(0)

## 获取采样点

将机械臂末端拖拽到9个点的位置，记录下机械臂末端的坐标值(x, y, z)

![](./image/p0.jpg)

In [6]:
p0 = arm.get_tool_pose()[:3]
p0

[251.0630547421913, 72.20832137789863, 1.5014849831005108]

![](./image/p1.jpg)

In [7]:
p1 = arm.get_tool_pose()[:3]
p1

[248.68614871237583, 0.8928881851349589, -3.578540516881702]

![](./image/p2.jpg)

In [8]:
p2 = arm.get_tool_pose()[:3]
p2

[251.58882856090514, -63.63533693367763, -0.8300457536466581]

![](./image/p3.jpg)

In [9]:
p3 = arm.get_tool_pose()[:3]
p3

[200.63797639493697, 64.79353777659162, -2.8416341352376833]

In [10]:
p4 = arm.get_tool_pose()[:3]
p4

[194.0664835549395, -0.0, -3.7291924971685546]

![](./image/p5.jpg)

In [11]:
p5 = arm.get_tool_pose()[:3]
p5

[197.1156715583039, -61.31949113805485, -0.641957080463925]

![](./image/p6.jpg)

In [12]:
p6 = arm.get_tool_pose()[:3]
p6

[149.63313936512978, 73.05521398577254, 4.929339708358896]

![](./image/p7.jpg)

In [13]:
p7 = arm.get_tool_pose()[:3]
p7

[149.71408314243652, 5.646786667476569, -0.7704939806262985]

![](./image/p8.jpg)

In [14]:
p8 = arm.get_tool_pose()[:3]
p8

[147.11274162597746, -59.779295335448225, 2.7807635917488653]

## 数据保存

In [15]:
ws_9p_at_arm = np.float32([p0, p1, p2, p3, p4, p5, p6, p7, p8])
ws_9p_at_arm

array([[251.063,  72.208,   1.501],
       [248.686,   0.893,  -3.579],
       [251.589, -63.635,  -0.83 ],
       [200.638,  64.794,  -2.842],
       [194.066,  -0.   ,  -3.729],
       [197.116, -61.319,  -0.642],
       [149.633,  73.055,   4.929],
       [149.714,   5.647,  -0.77 ],
       [147.113, -59.779,   2.781]], dtype=float32)

将数据存放到csv文件中

In [16]:
np.savetxt('config/arm9points.txt', ws_9p_at_arm, delimiter=',', fmt='%.1f')

## Base标定

In [17]:
# 设置Numpy的打印选项
# 精确位数3，不启用科学计数法
np.set_printoptions(precision=3, suppress=True)

# 载入九点标定法的配置文件
handeye_9points_config = None
with open("config/handeye_9points.yaml", 'r', encoding='utf-8') as f:
	handeye_9points_config = yaml.load(f.read(), Loader=yaml.SafeLoader)


# 载入机械臂末端达到九个点
# 载入九点在2D图像中的坐标
arm_9points = np.loadtxt("config/arm9points.txt", delimiter=',')
# 主要注意的是，对于关节臂构型的机械臂
# 有的在Z轴上的误差很大，因此需要将
# arm_9points里面Z轴坐标替换为Z轴的均值。
z_mean = np.mean(arm_9points[:, 2])

# z轴误差纠正拟合
# - 转化为r跟z的形式
r = np.sqrt(arm_9points[:, 0]**2 + arm_9points[:, 1]**2)
z_error =  arm_9points[:, 2] - z_mean
z_error_polyfit = np.polyfit(r, z_error, 1)
np.savetxt("config/z_error_polyfit.txt", z_error_polyfit, fmt='%.4f', delimiter=",")
arm_9points[:, 2] = z_mean
print(f"9点在机械臂坐标系的坐标: \n{arm_9points}")

# 9点在工作台坐标系的坐标
# ws_9points_df = pd.read_excel("config/九点坐标-工作台坐标系.xlsx")
# ws_9points = np.float64(ws_9points_df.to_numpy())[:, 1:]
w = handeye_9points_config["board_width"]
h = handeye_9points_config["board_height"]
# P0点的坐标
x0 = 0.5 * h
y0 = 0.5 * w
ws_9points = np.float64([
	[x0, y0, 0], 	#P0
	[x0, 0, 0], 	#P1
	[x0, -y0, 0], 	#P2
 	[0, y0, 0], 	#P3
	[0, 0, 0], 		#P4
	[0, -y0, 0], 	#P5
	[-x0, y0, 0], 	#P6
	[-x0, 0, 0], 	#P7
	[-x0, -y0, 0], 	#P8
])
print(f"9点在工作台坐标系的坐标: \n{ws_9points}")

# 求解机械臂基坐标系到工作台坐标系的变换
R, t = icp_solver_svd(arm_9points, ws_9points)
T_arm2ws = np.eye(4)
T_arm2ws[:3, :3] = R
T_arm2ws[:3, 3] = t.reshape(-1)
print("T_arm2ws: \n{}".format(T_arm2ws))
np.savetxt("config/T_arm2ws.txt", T_arm2ws, fmt='%.3f', delimiter=",")

9点在机械臂坐标系的坐标: 
[[251.1    72.2    -0.344]
 [248.7     0.9    -0.344]
 [251.6   -63.6    -0.344]
 [200.6    64.8    -0.344]
 [194.1    -0.     -0.344]
 [197.1   -61.3    -0.344]
 [149.6    73.1    -0.344]
 [149.7     5.6    -0.344]
 [147.1   -59.8    -0.344]]
9点在工作台坐标系的坐标: 
[[ 50.   66.5   0. ]
 [ 50.    0.    0. ]
 [ 50.  -66.5   0. ]
 [  0.   66.5   0. ]
 [  0.    0.    0. ]
 [  0.  -66.5   0. ]
 [-50.   66.5   0. ]
 [-50.    0.    0. ]
 [-50.  -66.5   0. ]]
T_arm2ws: 
[[  1.      0.02    0.    198.844]
 [ -0.02    1.      0.      3.544]
 [  0.      0.      1.     -0.344]
 [  0.      0.      0.      1.   ]]
